# 15 - Contraste de hipótesis
En este notebook contrastamos H1, H2 y H5 con la matriz de experimentos. Seguimos el criterio de la memoria:
- Una hipótesis se refuta en un caso cuando la F1 macro del conjunto de variables que se evalúa es inferior al 80 % de la del conjunto `completo` (`CLEARLY_INFERIOR_RATIO`), con el mismo modelo, la misma estrategia y el mismo perfil.
- La matriz con los hiperparámetros ajustados es la principal y la de los valores por defecto sirve de comprobación.
- Una conclusión es robusta si coincide en los cuatro modelos (el dummy es solo línea base y no se usa).

Cada hipótesis tiene 12 casos, 3 estrategias por 4 perfiles.

## Carga de los resultados

In [ ]:
import pandas as pd

from urban_accessibility_ml.config import MATRIX_MODELS, REPORTS_DIR
from urban_accessibility_ml.modeling.evaluate import cases_below, summarize_folds

CASE = ["strategy", "feature_set", "profile"]
MODELS = [m for m in MATRIX_MODELS if m != "dummy"]
# Conjunto que se evalúa en cada hipótesis (tab:comparaciones-hipotesis de la memoria)
HYPOTHESES = {"H1": "cartografico_aereo", "H2": "calle", "H5": "combinado"}

summary = {
    name: summarize_folds(pd.read_csv(REPORTS_DIR / "results" / f"experiment-matrix-{name}.csv"), CASE + ["model"])
    for name in ["tuned", "default"]
}
summary["tuned"].shape, summary["default"].shape

## Casos por debajo del 80 % de completo
Número de casos de 12 en los que cada hipótesis queda por debajo del 80 % de completo, por modelo y con cada matriz.
- H1 se evalúa con `cartografico_aereo`.
- H2 con `calle`.
- H5 con `combinado`.

In [ ]:
def count_below(summary):
    counts = pd.DataFrame({m: cases_below(summary, m)["feature_set"].value_counts() for m in MODELS})
    return counts.reindex(list(HYPOTHESES.values())).fillna(0).astype(int).set_axis(list(HYPOTHESES))


display(count_below(summary["tuned"]).rename_axis("ajustada"))
display(count_below(summary["default"]).rename_axis("por defecto"))

Observamos que H1, sobre cartográfico y aéreo:
- En la regresión logística, 9 de los 12 casos de la matriz ajustada quedan por debajo del 80 % de completo, mientras que sin ajustar son 4 de 12.
- En random_forest, XGBoost y LightGBM hay pocos casos por debajo, 3 o menos tanto en ajustada como por defecto.

H2, sobre datos del conjunto a pie de calle:
- En ajustada, la regresión logística tiene 3 casos de 12 por debajo del 80 %, así como 1 caso de LightGBM.
- Mientras que en por defecto, únicamente 1 caso de LightGBM está por debajo del 80 %.

En cuanto a H5, sobre ambos conjuntos de imágenes, se repiten los mismos valores que a pie de calle.
Como esta comprobación se ha hecho con el umbral del 50 %, recordemos que el conjunto de calle y combinado tenía las mismas dimensiones.

## Casos concretos

Vamos a comprobar todos los casos por cada una de las hipótesis.

La columna `por debajo en` indica si el caso específico está por debajo en las dos matrices (ajustada y por defecto), solo en la ajustada o solo en la de por defecto.

In [ ]:
def cases(hypothesis):
    feature_set = HYPOTHESES[hypothesis]
    parts = {}
    for model in MODELS:
        below = cases_below(summary["tuned"], model)[CASE].merge(
            cases_below(summary["default"], model)[CASE], on=CASE, how="outer", indicator="por debajo en"
        )
        below = below[below["feature_set"] == feature_set]
        below["por debajo en"] = below["por debajo en"].map(
            {"both": "ambas", "left_only": "solo ajustada", "right_only": "solo por defecto"}
        )
        keys = list(zip(below["strategy"], below["profile"]))
        for name, label in [("tuned", "ajustada"), ("default", "por defecto")]:
            data = summary[name][summary[name]["model"] == model]

            def f1(fs):
                rows = data[data["feature_set"] == fs].set_index(["strategy", "profile"])
                return rows["f1_macro_mean"].loc[keys].to_numpy()

            below[f"F1 macro ({label})"] = f1(feature_set)
            below[f"% de completo ({label})"] = 100 * f1(feature_set) / f1("completo")
        parts[model] = below[
            ["strategy", "profile", "F1 macro (ajustada)", "% de completo (ajustada)",
             "F1 macro (por defecto)", "% de completo (por defecto)", "por debajo en"]
        ]
    columns = pd.concat(parts, names=["modelo"]).reset_index(level="modelo").reset_index(drop=True)
    return columns.round({"F1 macro (ajustada)": 3, "F1 macro (por defecto)": 3, "% de completo (ajustada)": 1, "% de completo (por defecto)": 1})

### H1 (`cartografico_aereo`)


In [ ]:
cases("H1")


Se puede ver que más de la mitad de los casos son de regresión logística, concretamente:
- El perfil sensorial tiene la F1 macro tanto de ajustada como por defecto por debajo del 80 % del completo, en las estrategias cross_city y mixed; así como within_city en ajustada.
- La estrategia mixta solo está por encima del 80 % en por defecto en cognitiva y funcional. El resto está por debajo del 80 % en ambos casos.
- El perfil funcional está por debajo en ajustada en todas las estrategias.

En los modelos de árboles:
- El caso mixed con perfil cognitivo está por debajo del 80 % tanto en F1 macro ajustada como por defecto, en los tres modelos de árbol.
- random_forest tiene mixed global por debajo solo en el de por defecto.
- XGBoost tiene tanto cross_city sensorial como mixed global por debajo del 80 % en la ajustada.
- LightGBM tiene por debajo cross_city funcional tanto en ajustada como por defecto.


### H2 (`calle`)


In [ ]:
cases("H2")


Usando H2, del conjunto de calle, nos encontramos:
- En regresión logística, la F1 macro está por debajo del 80 % de completo, solo en ajustada de cross_city cognitiva y sensorial, y mixed funcional.
- En LightGBM hay un caso por debajo, cross_city global, en ambas matrices.

### H5 (`combinado`)


In [ ]:
cases("H5")


Coincide con los valores de H2, porque con el umbral del 50 %, las dimensiones de calle y combinado son las mismas.

## Robustez de cada caso

Vamos a ver, para cada hipótesis, qué ocurre en cada uno de sus 12 casos (estrategia y perfil) con los cuatro modelos. Reglas:
- Una hipótesis se considera refutada en un caso cuando lo está en los cuatro modelos, es decir, cuando la F1 macro queda por debajo del 80 % de completo en todos ellos.
- Se considera no refutada en un caso si no lo está por debajo en ningún modelo.
- En el resto de casos, se indicará que depende del modelo.

In [ ]:
def flags(summary, feature_set):
    """True si el caso (estrategia, perfil) queda por debajo del 80 %, por modelo."""
    idx = summary[summary["feature_set"] == feature_set].set_index(["strategy", "profile"]).index.unique()
    out = pd.DataFrame(False, index=idx, columns=MODELS)
    for m in MODELS:
        below = cases_below(summary, m)
        below = below[below["feature_set"] == feature_set]
        out.loc[list(zip(below["strategy"], below["profile"])), m] = True
    return out


def verdicts(f):
    n = f.sum(axis=1)
    return n.map(lambda k: "no refutada" if k == 0 else "refutada" if k == len(MODELS) else "depende del modelo")


SHORT = {"logistic_regression": "LR", "random_forest": "RF", "xgboost_balanced": "XGB", "lightgbm": "LGBM"}


def below_models(f):
    """Cuántos modelos dejan el caso por debajo y cuáles, por ejemplo "2 de 4 (LR, LGBM)"."""
    def label(row):
        names = [SHORT[m] for m in MODELS if row[m]]
        return f"{len(names)} de {len(MODELS)}" + (f" ({', '.join(names)})" if names else "")

    return f.apply(label, axis=1)


def robustness(hypothesis):
    feature_set = HYPOTHESES[hypothesis]
    columns = {}
    for name, label in [("tuned", "ajustada"), ("default", "por defecto")]:
        f = flags(summary[name], feature_set)
        columns[f"modelos por debajo ({label})"] = below_models(f)
        columns[f"veredicto ({label})"] = verdicts(f)
    table = pd.DataFrame(columns)
    display(table)
    display(table.filter(like="veredicto").apply(pd.Series.value_counts).fillna(0).astype(int))

### H1 (`cartografico_aereo`)


In [ ]:
robustness("H1")

Observamos que con la matriz ajustada solo hay 1 caso de 12 que se refuta en los cuatro modelos, que es mixed cognitiva.
Otros 3 casos no se refutan en ninguno (cross_city cognitiva y global, y within_city cognitiva), y los 8 restantes dependen del modelo.

En esos 8 casos que dependen del modelo, la regresión logística siempre está por debajo del 80 %. En 5 de ellos es el único modelo por debajo (mixed funcional y sensorial, y within_city funcional, global y sensorial), y en los otros 3 pasa también en XGBoost o LightGBM.

Con la matriz por defecto no hay ningún caso que se refute en los cuatro modelos. Son 6 los que no se refutan en ninguno y 6 los que dependen del modelo. En mixed cognitiva quedan por debajo `random_forest`, XGBoost y LightGBM, pero no así la regresión logística.

### H2 (`calle`)


In [ ]:
robustness("H2")

Observamos que con la matriz ajustada ningún caso se refuta en los cuatro modelos. 8 de 12 no se refutan en ninguno, y 4 dependen del modelo: cross_city cognitiva y sensorial y mixed funcional (solo con la regresión logística) y cross_city global (solo con LightGBM).

Con la matriz por defecto, 11 de 12 casos no se refutan en ningún modelo, y solo cross_city global (LightGBM) depende del modelo.

En 3 casos (cross_city cognitiva y sensorial, y mixed funcional) la regresión logística queda por debajo del 80 % de completo con la matriz ajustada, pero no con la de por defecto.

### H5 (`combinado`)


In [ ]:
robustness("H5")

Se repiten los mismos casos que en H2.

## Robustez frente al umbral

Vamos a ver cómo cambian los 12 casos de cada hipótesis al cambiar el umbral de viabilidad, con la matriz ajustada de `sensitivity.csv`.

Cada barra reparte los casos según cuántos de los cuatro modelos los dejan por debajo del 80 % de completo.

Los umbrales con un conjunto sin dimensiones aparecen como «vacío», y los que repiten el conjunto de otro umbral llevan una nota debajo.

In [ ]:
from urban_accessibility_ml.config import THRESHOLDS
from urban_accessibility_ml.preprocessing.features import sensitivity_sets

sensitivity = pd.read_csv(REPORTS_DIR / "results" / "sensitivity.csv")
sets, _ = sensitivity_sets()
# completo no depende del umbral, se toma de la matriz ajustada
completo = summary["tuned"][summary["tuned"]["feature_set"] == "completo"]
by_all = {
    t: pd.concat([completo, summarize_folds(sensitivity[sensitivity["threshold"] == t], CASE + ["model"])])
    for t in THRESHOLDS
}

In [ ]:
STATES = ["no refutada", "depende del modelo", "refutada"]


def counts(hypothesis):
    """Casos en cada estado por umbral. Un umbral cuya fila suma 0 es un conjunto vacío."""
    feature_set = HYPOTHESES[hypothesis]
    rows = {t: verdicts(flags(by_all[t], feature_set)).value_counts().reindex(STATES, fill_value=0) for t in THRESHOLDS}
    return pd.DataFrame(rows).T.rename_axis("umbral")


def threshold_label(hypothesis, t, sep=" "):
    """Etiqueta de un umbral, con aviso si el conjunto es vacío o repite el de un umbral anterior."""
    rows = sets[sets["feature_set"] == HYPOTHESES[hypothesis]]
    notice = ""
    if t not in rows["threshold"].values:
        notice = "vacío"
    else:
        first_seen = {}
        for u, dims in zip(rows["threshold"], rows["dimensions"]):
            first_seen.setdefault(dims, u)
            if u == t and first_seen[dims] != t:
                notice = f"={first_seen[dims]} %"
    return f"{t} %" + (f"{sep}({notice})" if notice else "")


counts("H1")

In [ ]:
import matplotlib.pyplot as plt

from urban_accessibility_ml.plotting.plots import save_figure, sequential_colors, use_style

use_style()
fig, axes = plt.subplots(1, len(HYPOTHESES), figsize=(10, 3.3), sharey=True)
for ax, hypothesis in zip(axes, HYPOTHESES):
    counts(hypothesis).plot(kind="bar", stacked=True, color=sequential_colors(len(STATES)), rot=0, ax=ax, legend=False)
    ax.set_title(f"{hypothesis} ({HYPOTHESES[hypothesis]})")
    for container, color in zip(ax.containers, ["black", "white", "white"]):
        numbers = [str(int(n)) if n else "" for n in container.datavalues]
        ax.bar_label(container, labels=numbers, label_type="center", color=color)
    ax.set_xlabel("umbral (%)")
    ax.set_xticklabels([threshold_label(hypothesis, t, "\n") for t in THRESHOLDS])
axes[0].set_ylabel("número de casos")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(labels), frameon=False)
fig.tight_layout(rect=(0, 0.08, 1, 1))
save_figure("robustez_umbral_barras", "0.15-contraste-hipotesis", fig=fig)
plt.show()

Observamos que en H1 el umbral del 40 % tiene 7 casos que no se refutan y 5 que dependen del modelo, sin ninguno refutado.
Con el 50 % y el 60 %, que tienen el mismo resultado por compartir dimensiones, hay 3, 8 y 1 (no refutados, dependen del modelo y refutado, respectivamente).
Con el 75 % y el 90 % el conjunto está vacío y no se puede contrastar H1.

En H2, tanto el 40 % como el 50 % tienen 8 casos que no se refutan y 4 que dependen del modelo, aunque no son exactamente los mismos casos.
Con el 60 % ningún caso queda por encima del 80 % en los cuatro modelos, en concreto, 11 dependen del modelo y 1 se refuta. El 75 % da el mismo resultado que el 60 %, porque tienen las mismas dimensiones.
Con el 90 % se refutan 8 casos en los cuatro modelos y 4 dependen del modelo.

En H5 es parecido, pero con el 60 % quedan 4 casos que no se refutan, 8 que dependen del modelo y ninguno refutado. Con el 75 % y el 90 % coincide con H2.
Parece que, en H2 y H5, cuanto más alto es el umbral, más casos dependen del modelo o se refutan, y que con el 40 % y el 50 % se mantienen en la mayoría de casos.

### Detalle por caso y umbral

Vamos a ver el detalle de cada caso (estrategia y perfil) con una figura por hipótesis. Cada celda indica cuántos de los cuatro modelos dejan el caso por debajo del 80 % de completo, con la matriz ajustada de `sensitivity.csv`. Un 0 significa que se mantiene en los cuatro y un 4 que se refuta en los cuatro.

Los conjuntos sin dimensiones aparecen en gris, y los umbrales que repiten el conjunto de otro umbral llevan una nota con ese umbral, por ejemplo «(=50 %)».

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from urban_accessibility_ml.config import PROFILES
from urban_accessibility_ml.plotting.plots import CMAP_SEC, save_figure, use_style

use_style()
STRATEGIES = ["within_city", "mixed", "cross_city"]
CASES = [(s, p) for s in STRATEGIES for p in PROFILES]


def models_below(hypothesis):
    """Modelos por debajo del 80 % en cada caso (columnas) y umbral (filas). NaN si el conjunto está vacío."""
    feature_set = HYPOTHESES[hypothesis]
    grid = pd.DataFrame(np.nan, index=THRESHOLDS, columns=pd.MultiIndex.from_tuples(CASES))
    for t in THRESHOLDS:
        if feature_set in by_all[t]["feature_set"].values:
            grid.loc[t] = flags(by_all[t], feature_set).sum(axis=1).reindex(CASES).to_numpy()
    return grid


def mapa(hypothesis):
    grid = models_below(hypothesis)
    text = grid.astype("Int64").astype(str).replace("<NA>", "")

    fig, ax = plt.subplots(figsize=(6.3, 2.8))
    ax.set_facecolor("#D9D9D9")  # gris, para los conjuntos vacíos
    sns.heatmap(
        grid, mask=grid.isna(), annot=text, fmt="", cmap=CMAP_SEC, vmin=0, vmax=len(MODELS),
        cbar=False, linewidths=1, linecolor="#CCCCCC", ax=ax,
    )
    ax.set_yticklabels([threshold_label(hypothesis, t) for t in THRESHOLDS], rotation=0)
    ax.set_xticklabels([profile for _, profile in CASES], rotation=90)
    ax.set_xlabel("")
    ax.set_ylabel("umbral")
    for i, strategy in enumerate(STRATEGIES):
        ax.text(i * 4 + 2, -0.3, strategy, ha="center", va="bottom", fontweight="bold")
    for x in (4, 8):
        ax.axvline(x, color="black", linewidth=1)
    ax.set_title(f"Número de modelos (de {len(MODELS)}) cuya F1 macro es inferior\nal 80 % de la del conjunto completo", pad=24)
    save_figure(f"robustez_umbral_{hypothesis}", "0.15-contraste-hipotesis", fig=fig)
    plt.show()

#### H1 (`cartografico_aereo`)

In [ ]:
mapa("H1")

Vemos que con el 50 % hay un caso en el que los cuatro modelos quedan por debajo del 80 %, que es mixed cognitiva. Con el 60 % pasa lo mismo, porque es el mismo conjunto que el 50 %.

Con el 40 % no hay ningún caso así. Lo máximo que se ve es un modelo por debajo, en within_city (todo menos cognitiva), y en mixed (global y sensorial).

Con el 50 % hay tres casos en los que ningún modelo baja del 80 %. Son within_city cognitiva y cross_city global y cognitiva.

En cambio, en mixed global y en cross_city funcional y sensorial hay dos modelos que quedan por debajo.

#### H2 (`calle`)

In [ ]:
mapa("H2")

Vemos que con el 40 % y el 50 % la mayoría de casos no tiene ningún modelo por debajo del 80 %.
Con el 40 %, cross_city tiene un modelo por debajo en los cuatro perfiles; mientras que con el 50 % lo tiene en global, sensorial y cognitiva, y también en mixed funcional.

Con el 60 % empiezan a aparecer más modelos por debajo. En mixed sensorial hay cuatro, en within_city sensorial hay tres y en within_city cognitiva hay dos. El 75 % da el mismo resultado que el 60 %.

Con el 90 %, los cuatro modelos quedan por debajo en todos los perfiles de within_city y de mixed. En cambio, en cross_city hay dos modelos por debajo en global y uno en el resto.

#### H5 (`combinado`)

In [ ]:
mapa("H5")

Vemos que con el 40 % y el 50 % la mayoría de casos no tiene ningún modelo por debajo del 80 %. Con el 40 %, cross_city tiene un modelo por debajo en global, sensorial y cognitiva. Con el 50 % pasa lo mismo, y además en mixed funcional.

Con el 60 % ningún caso tiene más de un modelo por debajo. En cambio, con el 75 % hay cuatro modelos por debajo en mixed sensorial, tres en within_city sensorial y dos en within_city cognitiva.

Con el 90 %, los cuatro modelos quedan por debajo en todos los perfiles de within_city y de mixed. En cross_city hay dos modelos por debajo en global y uno en el resto.